## Imports

In [48]:
import pandas as pd
import openai

## Read Intervention Data

In [62]:
df_interventions = pd.read_csv("../data/interventions.csv")
df_interventions.head()

,id,date_start,date_end,machine,machine_type,location,intervention_type,fault_code,fault_description,severity,subsystem,priority,technician,supervisor,duration_min,related_intervention,events,comments
0,INT-2022-0001,2022-01-01 06:13,2022-01-01 11:20,HX-350,Hydraulic Press,Shop A - Bay 3,PM,NaN,NaN,NaN,NaN,LOW,"Schmidt, K.","Chen, W.",307,NaN,Scheduled Quarterly system flush and pump effi...,PM Quarterly system flush and pump efficiency ...
1,INT-2022-0002,2022-01-01 07:24,2022-01-01 08:46,CB-200,Conveyor Belt System,Shop A - Transfer Line,CM,B-006,Carry roller bearing alarm,WARNING,Mechanical,MEDIUM,"Dubois, P.","Tanaka, Y.",82,NaN,Carry roller bearing alarm B-006. Elevated vib...,Fault B-006: roller #36 bearing failure. Vibra...
2,INT-2022-0003,2022-01-03 07:22,2022-01-03 09:02,CR-150,Cold Rolling Mill,Shop B - Line 2,PM,NaN,NaN,NaN,NaN,LOW,"Schmidt, K.","Kowalski, R.",100,NaN,Scheduled Weekly roll inspection and lubricati...,PM Weekly roll inspection and lubrication: tri...
3,INT-2022-0004,2022-01-03 06:14,2022-01-03 09:30,IH-300,Induction Hardening System,Shop D - HT Zone,CM,H-004,Frequency deviation >2%,WARNING,Electrical,MEDIUM,"Bauer, H.","Chen, W.",196,NaN,Frequency deviation alarm H-004. RF output fre...,Fault H-004: frequency deviation 0.07% from no...
4,INT-2022-0005,2022-01-04 07:57,2022-01-04 15:26,HX-200,Hydraulic Press,Shop A - Bay 1,PM,NaN,NaN,NaN,NaN,LOW,"Okonkwo, E.","Moreau, L.",449,NaN,Scheduled Quarterly system flush and pump effi...,PM Quarterly system flush and pump efficiency ...


In [71]:
def validate_fault_code(fault_code):
    if isinstance(fault_code, str) and len(fault_code.split('-')[0]) == 1 and len(fault_code.split('-')[1]) == 3:
        return True
    return False

In [72]:
df_interventions['is_valid_fault_code'] = df_interventions['fault_code'].apply(validate_fault_code)

## Preprocessing for Vector Store

In [73]:
df_interventions_cm = df_interventions[df_interventions["intervention_type"] == "CM"]

In [75]:
def create_summary(row):
    summary_parts = []
    
    if row.get('is_valid_fault_code') is True:
        summary_parts.append(f"[FAULT_CODE] {row['fault_code']}")
    
    if pd.notna(row.get('related_intervention')):
        summary_parts.append(f"[RELATED_INTERVENTION] {row['related_intervention']}")
        
    
    summary_parts.append(f"[EVENT] {row['events']}")
    summary_parts.append(f"[COMMENTS] {row['comments']}")
    
    return "\n".join(summary_parts).strip()


In [76]:
df_interventions_cm["summary"] = df_interventions_cm.apply(create_summary, axis=1)

/var/folders/qr/_t3nsks152d7njyq9d_kcqx40000gn/T/ipykernel_53532/1773380293.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_interventions_cm["summary"] = df_interventions_cm.apply(create_summary, axis=1)


## Sample 100 items for testing

In [79]:
df_interventions_cm_sample = df_interventions_cm.sample(n=100, random_state=42)

In [80]:
columns_to_keep = ["id", "date_start", "machine", "duration_min", "summary"]

data_to_embed = df_interventions_cm_sample[columns_to_keep].to_dict(orient='records')

### Define Embedding Function

In [83]:
import openai
import dotenv

dotenv.load_dotenv()

True

In [91]:
def embed_text(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [92]:
embedding_model = "text-embedding-3-small"

embeddings = [embed_text(record['summary'], model=embedding_model) for record in data_to_embed]

## Create Qdrant Collection

In [93]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [94]:
qdrant_host = "http://localhost:6333"

In [95]:
def create_or_get_collection(client, collection_name, vector_size):
    try:
        client.get_collection(collection_name)
        print(f"Collection '{collection_name}' already exists.")
    except Exception as e:
        print(f"Creating collection '{collection_name}'...")
        client.recreate_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
        )

In [132]:
collection_name = "cm_interventions"

In [ ]:
qdrant_client = QdrantClient(qdrant_host)

create_or_get_collection(qdrant_client, collection_name, vector_size=len(embeddings[0]))

Creating collection 'cm_interventions'...


/var/folders/qr/_t3nsks152d7njyq9d_kcqx40000gn/T/ipykernel_53532/4027101074.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(
/Users/jooaobrum/Library/CloudStorage/GoogleDrive-joao.paulo.brum14@gmail.com/My Drive/Projetos Pessoais/Projetos de Estudo/end2end-ai-engineering-bootcamp/hephaestus-agentic-maintenance/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:280: UserWarning: Qdrant client version 1.17.1 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


In [106]:
import uuid

In [112]:
def upsert_points(client, collection_name, records, embeddings, metadata):
    points = []
    for record, embedding in zip(records, embeddings):
        point = PointStruct(
            id=str(uuid.uuid4()),
            vector=embedding,
            payload={
                "id": record['id'],
                "date_start": record['date_start'],
                "machine": record['machine'],
                "duration_min": record['duration_min'],
                "summary": record['summary'],
                "embedding_model": metadata.get("embedding_model", ""),
                "created_at": metadata.get("created_at", "")
            }
        )
        points.append(point)
    
    client.upsert(collection_name=collection_name, points=points)
    print(f"Upserted {len(points)} points into collection '{collection_name}'.")


In [115]:
data_to_embed[0]

{'id': 'INT-2024-0226',
 'date_start': '2024-03-29 22:09',
 'machine': 'CB-200',
 'duration_min': 239,
 'summary': '[FAULT_CODE] B-001\n[EVENT] Belt misalignment alarm B-001. Laser sensor BAL-101 reading 1.4 mm off-centre (limit 15 mm).\n[COMMENTS] Fault B-001: belt tracking 1.4 mm off-centre. Root cause: cable abrasion at conduit entry point from long-term vibration. LOTO applied per SP-LOTO-001. Faulty component replaced. System function-tested before returning to production. Belt re-tracked to 0.05 mm. Training idler locked. Spare parts replenished in stores. Stock level confirmed.'}

In [113]:
metadata_info = {"embedding_model": embedding_model, "created_at": pd.Timestamp.now().isoformat()}

In [114]:
upsert_points(qdrant_client, collection_name, data_to_embed, embeddings, metadata_info)

Upserted 100 points into collection 'cm_interventions'.


## Define Function for Querying the Vector Database

In [130]:
def retrieve_data(client, collection_name, query_vector, top_k=5):
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k
    )
    return search_results.points


In [131]:
query_text = "Pump vibration issue on machine HX-200 causing downtime"

query_embedding = embed_text(query_text, model=embedding_model)

search_results = retrieve_data(qdrant_client, collection_name, query_embedding, top_k=5)

for result in search_results:
    print(f"ID: {result.id}, Score: {result.score}")
    print(f"Payload: {result.payload}")
    print("-" * 50)

ID: d6432b0f-44a3-457d-b289-746de4be32a2, Score: 0.6567029
Payload: {'id': 'INT-2024-0612', 'date_start': '2024-08-03 14:36', 'machine': 'HX-350', 'duration_min': 276, 'summary': '[FAULT_CODE] E-010\n[EVENT] Vibration monitoring system triggered E-010. Pump bearing vibration 880 mm/s RMS, above 3.5 mm/s threshold. Production continued at reduced rate pending investigation.\n[COMMENTS] Fault E-010: vibration 880 mm/s RMS at pump bearing. Spectrum analysis shows dominant 1× running speed component. Root cause: cable abrasion at conduit entry point from long-term vibration. Root cause isolated and corrected. Replaced associated consumable as preventive measure. Machine re-certified. Post-repair vibration: 0.3 mm/s RMS. Trending added to weekly checks. Machine returned to production. Monitoring interval increased for next 4 weeks.', 'embedding_model': 'text-embedding-3-small', 'created_at': '2026-03-28T11:20:38.230911'}
--------------------------------------------------
ID: a2f9009f-17bf-4